# Client-Controlled Cyclic Weight Transfer with CCWFCyclicRecipe

This notebook shows how to run a [Cyclic Weight Transfer](https://pubmed.ncbi.nlm.nih.gov/29617797) (CWT) job using the `CCWFCyclicRecipe` — NVFlare's declarative recipe API for the Client-Controlled Cyclic Workflow (CCWF).

If you are new to the CCWF Cyclic Workflow itself, start with [cyclic_weight_transfer_example.ipynb](./cyclic_weight_transfer_example.ipynb), which builds the same job manually using the raw Job API. This notebook focuses on the recipe layer: what it replaces, what it exposes, and when to reach for it.

## What is CCWFCyclicRecipe?

`CCWFCyclicRecipe` is a single declarative object that wires the full CCWF cyclic stack:

| Component wired automatically | Role |
|---|---|
| `CCWFJob` | Job container (replaces `FedJob` for CCWF) |
| `CyclicServerController` | Server: counts rounds, picks starting client, coordinates the ring |
| `CyclicClientController` | Client: runs the ring protocol — receives model, triggers training, forwards model |
| `PTFileModelPersistor` | Stores the model on each client (not on the server) |
| `SimpleModelShareableGenerator` | Serialises/deserialises the model for peer-to-peer transfer |
| `ScriptRunner` / `PTInProcessClientAPIExecutor` | Runs your training script on each client |

Without the recipe, all of these must be assembled and connected by hand — about 20 lines of boilerplate.
With the recipe, the same job takes 8 lines.

## CCWF Cyclic vs. Swarm Learning

Both are CCWF variants where the server never touches model weights.
The difference is what happens at each client:

| | Cyclic (this notebook) | Swarm Learning |
|---|---|---|
| Training order | Sequential (one at a time) | Parallel (all at once) |
| Aggregation | None — model is passed directly | Weighted average by a peer aggregator |
| Analogy | Relay race — one baton passed in order | FedAvg — but aggregation done by a client |

## Diagram

<img src="figs/cyclic_ccwf.png" alt="CCWF cyclic topology" width="35%" />

The server assigns a starting client and communicates only coordination messages (`cyclic_config`, `cyclic_start`). Model weights travel exclusively along the peer-to-peer ring, encrypted end-to-end.

## Install requirements

In [ ]:
!pip install --upgrade pip
!pip install -r ./requirements.txt

## Prepare data

Download CIFAR-10 and copy the shared training script and model definition.

In [ ]:
!python ../download_cifar10.py
!cp ../train.py train.py
!cp ../net.py net.py

SMOKE_TRAIN_SIZE = 256
SMOKE_TEST_SIZE = 256
SMOKE_TRAIN_ARGS = (
    f"--local_epochs 1 --batch_size 32 --num_workers 0 "
    f"--train_size {SMOKE_TRAIN_SIZE} --test_size {SMOKE_TEST_SIZE}"
)

from pathlib import Path

train_path = Path("train.py")
train_source = train_path.read_text()
train_source = train_source.replace("best_accuracy = 0.0", "best_accuracy = -1.0")
train_source = train_source.replace(
    '    parser.add_argument("--local_epochs", type=int, default=2, nargs="?")\n',
    '    parser.add_argument("--local_epochs", type=int, default=2, nargs="?")\n'
    '    parser.add_argument("--train_size", type=int, default=None, nargs="?")\n'
    '    parser.add_argument("--test_size", type=int, default=None, nargs="?")\n',
)
train_source = train_source.replace(
    "    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)\n"
    "    testset = torchvision.datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)\n"
    "    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n",
    "    if args.train_size is not None:\n"
    "        trainset = torch.utils.data.Subset(trainset, range(args.train_size))\n"
    "    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)\n"
    "    testset = torchvision.datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)\n"
    "    if args.test_size is not None:\n"
    "        testset = torch.utils.data.Subset(testset, range(args.test_size))\n"
    "    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)\n",
)
train_source = train_source.replace("Accuracy on the 10000 test images", "Accuracy on the {len(testloader.dataset)} test images")
train_path.write_text(train_source)

import atexit, os
atexit.register(lambda: os.remove('train.py') if os.path.exists('train.py') else None)
atexit.register(lambda: os.remove('net.py') if os.path.exists('net.py') else None)

## Raw Job API (for comparison)

This is the full manual wiring required without the recipe.  It is shown here only for comparison — you do **not** need to run this cell.

In [ ]:
# --- RAW JOB API (shown for comparison only) ---
from net import Net

from nvflare.app_common.ccwf.ccwf_job import CCWFJob, CyclicClientConfig, CyclicServerConfig
from nvflare.app_common.ccwf.comps.simple_model_shareable_generator import SimpleModelShareableGenerator
from nvflare.app_opt.pt.file_model_persistor import PTFileModelPersistor
from nvflare.job_config.script_runner import ScriptRunner

n_clients = 2
num_rounds = 1
train_script = "train.py"

job = CCWFJob(name="cifar10_cyclic")

job.add_cyclic(
    server_config=CyclicServerConfig(num_rounds=num_rounds, max_status_report_interval=300),
    client_config=CyclicClientConfig(
        executor=ScriptRunner(script=train_script, script_args=SMOKE_TRAIN_ARGS),
        persistor=PTFileModelPersistor(model=Net()),
        shareable_generator=SimpleModelShareableGenerator(),
    ),
)

job.simulator_run("/tmp/nvflare/jobs/workdir/pt_cyclic", n_clients=n_clients, gpu="0")

## Recipe API

`CCWFCyclicRecipe` replaces all of the manual wiring above.
The call signature mirrors `FedAvgRecipe` and `SwarmLearningRecipe`:

- Pass your model and training script — the recipe wires the persistor, shareable generator, and both controllers.
- Pass a `SimEnv` to run locally, or a `PocEnv` / `ProdEnv` to submit to a real deployment.
- Call `add_experiment_tracking` after construction to attach a TensorBoard or MLflow receiver.

### Why `client_side=True, server_side=False` for tracking

`CCWFJob` does not include a `ConvertToFedEvent` bridge (that is a `BaseFedJob` feature).  Without it, analytics events generated during training stay on the client and never reach the server.  Client-side tracking is therefore the correct choice here — each client writes its own TensorBoard logs locally.

> **Quick-start note:** The cells below use `num_rounds=1` and pass `SMOKE_TRAIN_ARGS` to limit each client to 256 train and 256 test samples.  For real training, increase `num_rounds` (e.g. 5–10) and remove `train_args` to use the script's defaults (`batch_size=4`, `local_epochs=2`).

In [ ]:
from net import Net

from nvflare.app_opt.pt.recipes.ccwf_cyclic import CCWFCyclicRecipe
from nvflare.recipe import SimEnv, add_experiment_tracking

n_clients = 2
num_rounds = 1

recipe = CCWFCyclicRecipe(
    name="cifar10_cyclic_cc",
    min_clients=n_clients,
    num_rounds=num_rounds,
    model=Net(),
    train_script="train.py",
    train_args=SMOKE_TRAIN_ARGS,
)

add_experiment_tracking(recipe, tracking_type="tensorboard", client_side=True, server_side=False)

env = SimEnv(
    num_clients=n_clients,
    workspace_root="/tmp/nvflare/jobs/workdir/pt_cyclic_cc_recipe",
    gpu_config="0",
)

run = recipe.execute(env)
print("Job Status:", run.get_status())
print("Results at:", run.get_result())

## Key parameters

| Parameter | Default | Description |
|---|---|---|
| `model` | — | `nn.Module` instance, or a `{"class_path": ..., "args": ...}` dict config |
| `train_script` | — | Path to the client training script (Client API: `flare.init()` / `flare.receive()` / `flare.send()`) |
| `num_rounds` | — | Number of cyclic rounds; one round = one full pass around all clients |
| `min_clients` | — | Minimum number of clients required to start the job |
| `train_args` | `""` | Extra CLI arguments forwarded to `train_script` |
| `initial_ckpt` | `None` | Checkpoint to bootstrap from. Relative paths are bundled; absolute paths resolved server-side |
| `cyclic_order` | `"fixed"` | Ring order: `"fixed"`, `"random"`, or `"random_without_same_in_a_row"` |
| `starting_client` | `""` | Name of the first client in the ring; empty string lets the server choose |
| `do_cross_site_eval` | `False` | Append a CCWF cross-site evaluation phase after training completes |
| `max_status_report_interval` | `300` | Seconds between client status reports |
| `launch_external_process` | `False` | Run `train_script` in a subprocess instead of in-process |

## Optional: cross-site evaluation

Pass `do_cross_site_eval=True` to append a CCWF cross-site evaluation phase directly after training.  The same `train_script` must handle the `validate` and `submit_model` tasks via `flare.is_evaluate()` and `flare.is_submit_model()`.

In [ ]:
recipe_with_cse = CCWFCyclicRecipe(
    name="cifar10_cyclic_cc_cse",
    min_clients=n_clients,
    num_rounds=num_rounds,
    model=Net(),
    train_script="train.py",
    train_args=SMOKE_TRAIN_ARGS,
    do_cross_site_eval=True,
    cross_site_eval_timeout=300,
)

env_cse = SimEnv(
    num_clients=n_clients,
    workspace_root="/tmp/nvflare/jobs/workdir/pt_cyclic_cc_recipe_cse",
    gpu_config="0",
)

run_cse = recipe_with_cse.execute(env_cse)
print("Job Status:", run_cse.get_status())
print("Results at:", run_cse.get_result())

## Summary

`CCWFCyclicRecipe` reduces the CCWF cyclic job to a single declarative call:

- The **server** runs `CyclicServerController`: counts rounds and coordinates the ring.  It never receives model weights.
- Each **client** runs `CyclicClientController` (ring protocol) backed by `PTFileModelPersistor` and `SimpleModelShareableGenerator`.  Your `train_script` is invoked for the `train` task each time the model arrives at that client.
- Model weights travel exclusively over an encrypted peer-to-peer channel — the server sees only coordination messages.
- Pass `do_cross_site_eval=True` to add a CCWF cross-site evaluation phase automatically after training.

### Relationship to the raw Job API

The recipe is a thin wrapper around `CCWFJob.add_cyclic()`.  Everything it wires is visible in the exported job config at `<workspace_root>/<name>/`.  If you need component-level control not exposed by the recipe (e.g. a custom shareable generator or a non-standard cyclic order), drop down to the raw API shown in [cyclic_weight_transfer_example.ipynb](./cyclic_weight_transfer_example.ipynb).

### Next steps

- [Swarm Learning](../07.2.3_swarm_learning/swarm_learning.ipynb) — the parallel-training CCWF variant where a peer client aggregates updates instead of relaying them.
- [Advanced algorithms](../07.2.1_advanced_algos/) — FedProx, Scaffold, FedOpt, and other server-controlled variants.
- [NVFlare CCWF docs](https://nvflare.readthedocs.io/en/main/programming_guide/controllers/client_controlled_workflows.html) — full reference for all CCWF controllers and parameters.